# Tata Technologies Ltd. - TechPulse FY-26: Applied AI & ML
## Lab Statement 4: Vehicle Price Prediction

**Curriculum Context:** Unit 2 – Machine Learning & Applications (Case Study: Car Price Prediction)  
**Track:** AI & ML | **Level:** Intermediate  
**Dataset:** Structured Automotive Resale Valuation Dataset (`car_price_dataset.csv`)  
**Domain Focus:** Automotive Trade-In Pricing, Residual Value Forecasting & Multi-Factor Depreciation Analytics  

---

### 🎯 Learning Objectives
1. Model economic depreciation and resale value decay in used vehicles.
2. Build Scikit-Learn `ColumnTransformer` pipelines combining scaling and encoding.
3. Fit and compare Linear Regression, Ridge, Lasso, Decision Trees, Random Forests, and Gradient Boosting.
4. Evaluate regression performance using MAE (₹ Lakhs), RMSE, MAPE (%), and $R^2$.
5. Interpret feature importances and pricing sensitivities across vehicle attributes.

---

### Step 0: Imports & Library Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.dpi'] = 120
print("Environment initialized!")

### Step 1: Ingesting & Exploring Structured Car Resale Dataset

In [ ]:
df = pd.read_csv('car_price_dataset.csv')
print(f"Records: {df.shape[0]} | Attributes: {df.shape[1]}")
df.head()

In [ ]:
df.info()
print("\nDescriptive Summary:")
df.describe()

### Step 2: Exploratory Data Analysis & Depreciation Visualizations

In [ ]:
from IPython.display import Image
Image('plots/01_price_distribution_and_correlation.png')

In [ ]:
Image('plots/02_scatter_key_features_vs_price.png')

### Step 3: Preprocessing Pipeline & Partitioning

In [ ]:
numeric_features = ['age_years', 'present_price_lakhs', 'kms_driven', 'owner_count']
categorical_features = ['fuel_type', 'seller_type', 'transmission']

X = df[numeric_features + categorical_features]
y = df['selling_price_lakhs']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
print(f"Train size: {X_train.shape[0]} | Test size: {X_test.shape[0]}")

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), categorical_features)
    ]
)
preprocessor.fit(X_train)
print("Preprocessor fitted without data leakage!")

### Step 4: Model Training & Comparative Benchmarking

In [ ]:
models = {
    'Multiple Linear Regr': Pipeline([('prep', preprocessor), ('reg', LinearRegression())]),
    'Ridge Regression (L2)': Pipeline([('prep', preprocessor), ('reg', RidgeCV(alphas=np.logspace(-3, 3, 25), cv=5))]),
    'Lasso Regression (L1)': Pipeline([('prep', preprocessor), ('reg', LassoCV(alphas=np.logspace(-4, 1, 25), cv=5, max_iter=3000, random_state=42))]),
    'Decision Tree Regr': Pipeline([('prep', preprocessor), ('reg', DecisionTreeRegressor(max_depth=6, min_samples_split=8, random_state=42))]),
    'Random Forest Regr': Pipeline([('prep', preprocessor), ('reg', RandomForestRegressor(n_estimators=100, max_depth=10, min_samples_split=5, random_state=42, n_jobs=1))]),
    'Gradient Boosting Regr': Pipeline([('prep', preprocessor), ('reg', GradientBoostingRegressor(n_estimators=120, learning_rate=0.08, max_depth=4, random_state=42))])
}

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mape = mean_absolute_percentage_error(y_test, y_pred) * 100.0
    r2 = r2_score(y_test, y_pred)
    results.append({'Model': name, 'Test MAE (₹ L)': round(mae, 3), 'Test RMSE (₹ L)': round(rmse, 3), 'Test MAPE (%)': round(mape, 2), 'Test R²': round(r2, 4)})

pd.DataFrame(results)

### Step 5: Visual Diagnostics & Feature Importances

In [ ]:
Image('plots/03_model_evaluation_comparison.png')

In [ ]:
Image('plots/04_feature_importance_ranking.png')

### Step 6: Interactive Resale Price Predictor

In [ ]:
best_model = models['Gradient Boosting Regr']

def estimate_car_value(age_years, present_price_lakhs, kms_driven, fuel_type, seller_type, transmission, owner_count=0):
    input_df = pd.DataFrame([{
        'age_years': age_years,
        'present_price_lakhs': present_price_lakhs,
        'kms_driven': kms_driven,
        'fuel_type': fuel_type,
        'seller_type': seller_type,
        'transmission': transmission,
        'owner_count': owner_count
    }])
    pred_lakhs = best_model.predict(input_df)[0]
    print(f"Estimated Resale Value: ₹{pred_lakhs:.2f} Lakhs (₹{pred_lakhs * 100000:,.0f})")

# Example: 5-year-old Tata Nexon Diesel Manual with 55,000 km, showroom price ₹12.0 L
estimate_car_value(age_years=5, present_price_lakhs=12.0, kms_driven=55000, fuel_type='Diesel', seller_type='Individual', transmission='Manual')